In [5]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

MAX_FEATURES  = 10000
MAX_LEN       = 200
EMBEDDING_DIM = 128

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=MAX_FEATURES)
X_train = pad_sequences(X_train, maxlen=MAX_LEN)
X_test  = pad_sequences(X_test,  maxlen=MAX_LEN)

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

rnn_model = Sequential([
    Embedding(MAX_FEATURES, EMBEDDING_DIM),
    Dropout(0.3),
    SimpleRNN(64),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
rnn_model.fit(X_train, y_train, epochs=20, batch_size=64, validation_split=0.2, callbacks=[early_stop])

rnn_loss, rnn_acc = rnn_model.evaluate(X_test, y_test)
print(f"SimpleRNN Test Accuracy: {rnn_acc:.4f}")
rnn_model.save('simple_rnn_model.h5')

early_stop2 = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

lstm_model = Sequential([
    Embedding(MAX_FEATURES, EMBEDDING_DIM),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.fit(X_train, y_train, epochs=20, batch_size=64, validation_split=0.2, callbacks=[early_stop2])

lstm_loss, lstm_acc = lstm_model.evaluate(X_test, y_test)
print(f"LSTM Test Accuracy: {lstm_acc:.4f}")
lstm_model.save('lstm_model.h5')

Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 27ms/step - accuracy: 0.5454 - loss: 0.6828 - val_accuracy: 0.6400 - val_loss: 0.6254
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.7743 - loss: 0.4889 - val_accuracy: 0.6900 - val_loss: 0.5834
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.8940 - loss: 0.2766 - val_accuracy: 0.7668 - val_loss: 0.5403
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9204 - loss: 0.2103 - val_accuracy: 0.7846 - val_loss: 0.5571
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9753 - loss: 0.0826 - val_accuracy: 0.7674 - val_loss: 0.7022
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7637 - loss: 0.5427


SimpleRNN Test Accuracy: 0.7640
Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.6871 - loss: 0.5574 - val_accuracy: 0.8674 - val_loss: 0.3175
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9059 - loss: 0.2539 - val_accuracy: 0.8622 - val_loss: 0.3695
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9339 - loss: 0.1806 - val_accuracy: 0.8578 - val_loss: 0.3503
782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8655 - loss: 0.3202


LSTM Test Accuracy: 0.8667


In [7]:
!pip install fastapi uvicorn nest-asyncio -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import nest_asyncio
import uvicorn
import subprocess
import threading
import time
import re
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
from pydantic import BaseModel

nest_asyncio.apply()

MAX_FEATURES = 10000
MAX_LEN      = 200

rnn_model  = load_model('simple_rnn_model.h5')
lstm_model = load_model('lstm_model.h5')

word_index = imdb.get_word_index()
word_index = {k: (v + 3) for k, v in word_index.items()}
word_index["<PAD>"]    = 0
word_index["<START>"]  = 1
word_index["<UNK>"]    = 2
word_index["<UNUSED>"] = 3

def text_to_sequence(text):
    tokens   = text.lower().split()
    sequence = [word_index.get(w, 2) for w in tokens]
    sequence = [i if i < MAX_FEATURES else 2 for i in sequence]
    return pad_sequences([sequence], maxlen=MAX_LEN)

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class ReviewRequest(BaseModel):
    text: str

@app.post("/predict")
def predict(req: ReviewRequest):
    seq        = text_to_sequence(req.text)
    rnn_score  = float(rnn_model.predict(seq,  verbose=0)[0][0])
    lstm_score = float(lstm_model.predict(seq, verbose=0)[0][0])
    return {
        "rnn_result":  {"score": round(rnn_score,  4), "label": "Positive 😊" if rnn_score  >= 0.5 else "Negative 😞"},
        "lstm_result": {"score": round(lstm_score, 4), "label": "Positive 😊" if lstm_score >= 0.5 else "Negative 😞"}
    }

HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>RNN vs LSTM Sentiment Analyzer</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    font-family: 'Segoe UI', sans-serif;
    background: linear-gradient(135deg, #0f0c29, #302b63, #24243e);
    min-height: 100vh;
    display: flex;
    align-items: center;
    justify-content: center;
    padding: 30px;
  }
  .container {
    background: rgba(255,255,255,0.05);
    backdrop-filter: blur(12px);
    border: 1px solid rgba(255,255,255,0.1);
    border-radius: 20px;
    padding: 40px;
    width: 100%;
    max-width: 860px;
    color: #fff;
  }
  h1 {
    text-align: center;
    font-size: 2rem;
    margin-bottom: 6px;
    background: linear-gradient(90deg, #a78bfa, #60a5fa);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
  }
  .subtitle {
    text-align: center;
    color: rgba(255,255,255,0.45);
    margin-bottom: 30px;
    font-size: 0.95rem;
  }
  textarea {
    width: 100%;
    height: 140px;
    padding: 16px;
    background: rgba(255,255,255,0.08);
    border: 1px solid rgba(255,255,255,0.15);
    border-radius: 12px;
    color: #fff;
    font-size: 1rem;
    resize: vertical;
    outline: none;
    transition: border 0.3s;
  }
  textarea:focus { border-color: #a78bfa; }
  textarea::placeholder { color: rgba(255,255,255,0.3); }
  button {
    width: 100%;
    margin-top: 16px;
    padding: 14px;
    background: linear-gradient(90deg, #a78bfa, #60a5fa);
    border: none;
    border-radius: 12px;
    color: #fff;
    font-size: 1.1rem;
    font-weight: 600;
    cursor: pointer;
    transition: opacity 0.3s;
  }
  button:hover { opacity: 0.85; }
  button:disabled { opacity: 0.5; cursor: not-allowed; }
  .cards {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 20px;
    margin-top: 30px;
  }
  .card {
    background: rgba(255,255,255,0.06);
    border: 1px solid rgba(255,255,255,0.1);
    border-radius: 16px;
    padding: 24px;
    text-align: center;
    transition: transform 0.3s;
  }
  .card:hover { transform: translateY(-4px); }
  .card h2 { font-size: 1.2rem; margin-bottom: 16px; color: #c4b5fd; }
  .score-ring {
    width: 110px;
    height: 110px;
    border-radius: 50%;
    margin: 0 auto 16px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 1.6rem;
    font-weight: 700;
    border: 4px solid rgba(255,255,255,0.1);
    background: rgba(255,255,255,0.05);
    transition: border-color 0.5s, color 0.5s;
  }
  .positive { border-color: #4ade80 !important; color: #4ade80 !important; }
  .negative { border-color: #f87171 !important; color: #f87171 !important; }
  .label { font-size: 1.15rem; font-weight: 600; margin-bottom: 8px; }
  .bar-wrap {
    background: rgba(255,255,255,0.1);
    border-radius: 999px;
    height: 8px;
    margin-top: 12px;
    overflow: hidden;
  }
  .bar-fill {
    height: 100%;
    border-radius: 999px;
    width: 0%;
    transition: width 1s ease;
  }
  .positive-bar { background: linear-gradient(90deg, #4ade80, #22d3ee); }
  .negative-bar { background: linear-gradient(90deg, #f87171, #fb923c); }
  .hidden { display: none; }
  .spinner {
    width: 40px; height: 40px;
    border: 4px solid rgba(255,255,255,0.15);
    border-top-color: #a78bfa;
    border-radius: 50%;
    animation: spin 0.8s linear infinite;
    margin: 20px auto 0;
  }
  @keyframes spin { to { transform: rotate(360deg); } }
</style>
</head>
<body>
<div class="container">
  <h1>🎬 RNN vs LSTM</h1>
  <p class="subtitle">Sentiment Analyzer — Type a movie review and compare both models</p>
  <textarea id="reviewText" placeholder="e.g. This movie was absolutely fantastic! The acting was superb..."></textarea>
  <button id="analyzeBtn" onclick="analyze()">⚡ Analyze Sentiment</button>
  <div id="spinner" class="spinner hidden"></div>
  <div class="cards hidden" id="cards">
    <div class="card">
      <h2>🔁 SimpleRNN</h2>
      <div class="score-ring" id="rnnRing">—</div>
      <div class="label" id="rnnLabel">—</div>
      <div class="bar-wrap"><div class="bar-fill" id="rnnBar"></div></div>
    </div>
    <div class="card">
      <h2>🧠 LSTM</h2>
      <div class="score-ring" id="lstmRing">—</div>
      <div class="label" id="lstmLabel">—</div>
      <div class="bar-wrap"><div class="bar-fill" id="lstmBar"></div></div>
    </div>
  </div>
</div>
<script>
  async function analyze() {
    const text = document.getElementById("reviewText").value.trim();
    if (!text) { alert("Please enter a review!"); return; }
    const btn     = document.getElementById("analyzeBtn");
    const spinner = document.getElementById("spinner");
    const cards   = document.getElementById("cards");
    btn.disabled = true;
    spinner.classList.remove("hidden");
    cards.classList.add("hidden");
    try {
      const res  = await fetch("/predict", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify({ text })
      });
      const data = await res.json();
      renderResult("rnn",  data.rnn_result);
      renderResult("lstm", data.lstm_result);
      cards.classList.remove("hidden");
    } catch(e) {
      alert("Error: " + e.message);
    } finally {
      btn.disabled = false;
      spinner.classList.add("hidden");
    }
  }
  function renderResult(prefix, result) {
    const isPos = result.label.includes("Positive");
    const ring  = document.getElementById(prefix + "Ring");
    const label = document.getElementById(prefix + "Label");
    const bar   = document.getElementById(prefix + "Bar");
    ring.textContent  = (result.score * 100).toFixed(1) + "%";
    ring.className    = "score-ring " + (isPos ? "positive" : "negative");
    label.textContent = result.label;
    label.style.color = isPos ? "#4ade80" : "#f87171";
    bar.className = "bar-fill " + (isPos ? "positive-bar" : "negative-bar");
    setTimeout(() => { bar.style.width = (result.score * 100) + "%"; }, 100);
  }
  document.getElementById("reviewText").addEventListener("keydown", function(e) {
    if (e.ctrlKey && e.key === "Enter") analyze();
  });
</script>
</body>
</html>
"""

@app.get("/", response_class=HTMLResponse)
def frontend():
    return HTML

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

process = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

for line in process.stdout:
    line = line.decode()
    match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if match:
        print(f"\n✅ App is live at: {match.group(0)}\n")
        break

INFO:     Started server process [1038]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



✅ App is live at: https://contacting-answered-missile-opt.trycloudflare.com



In [8]:
!git config --global user.email "khadijalizadehh@gmail.com"
!git config --global user.name "KhadijaAlizadaa"

!git init
!git add simple_rnn_model.h5 lstm_model.h5
!echo "# RNN vs LSTM Sentiment Analyzer" > README.md
!git add README.md
!git commit -m "RNN vs LSTM Sentiment Analyzer project"
!git branch -M main
!git remote add origin https://github.com
!git push -u origin main

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
[master (root-commit) 9edc847] RNN vs LSTM Sentiment Analyzer project
 3 files changed, 1 insertion(+)
 create mode 100644 README.md
 create mode 100644 lstm_model.h5
 create mode 100644 simple_rnn_model.h5
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (5/5), 27.97 MiB | 10.10 MiB/s, done.
Total 5 (delta 0), reused 0 (delta 0), pack-reused 0